# Atividade 04 — SVM (Máquinas de Vetores de Suporte)

**Tema:** classificar **símbolos gráficos** (estrela, cruz e losango) a partir de imagens sintéticas.

O SVM não recebe a imagem diretamente: cada pixel vira um atributo numérico.

> Compatível com Google Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

In [ ]:
TAMANHO = 32
AMOSTRAS = 80
CLASSES = ["estrela", "cruz", "losango"]
ROTULOS = {n: i for i, n in enumerate(CLASSES)}

def desenhar_estrela(t, rng):
    img = np.zeros((t, t), dtype=np.float32)
    cx, cy = t // 2, t // 2
    r_ext, r_int = rng.integers(t//5, t//3), rng.integers(t//8, t//6)
    angulos = np.linspace(0, 2*np.pi, 11)[:-1]
    pts = []
    for i, a in enumerate(angulos):
        r = r_ext if i % 2 == 0 else r_int
        pts.append([int(cx + r*np.cos(a)), int(cy + r*np.sin(a))])
    pts = np.array(pts)
    for y in range(t):
        for x in range(t):
            if _ponto_em_poligono(x, y, pts):
                img[y, x] = 1.0
    return img

def _ponto_em_poligono(x, y, pts):
    n = len(pts)
    dentro = False
    j = n - 1
    for i in range(n):
        yi, xi = pts[i][1], pts[i][0]
        yj, xj = pts[j][1], pts[j][0]
        if ((yi > y) != (yj > y)) and (x < (xj - xi) * (y - yi) / (yj - yi + 1e-9) + xi):
            dentro = not dentro
        j = i
    return dentro

def desenhar_cruz(t, rng):
    img = np.zeros((t, t), dtype=np.float32)
    esp = rng.integers(3, 6)
    meio = t // 2
    img[meio-esp:meio+esp+1, :] = 1.0
    img[:, meio-esp:meio+esp+1] = 1.0
    return img

def desenhar_losango(t, rng):
    img = np.zeros((t, t), dtype=np.float32)
    meio = t // 2
    h = rng.integers(t//5, t//3)
    w = rng.integers(t//5, t//3)
    pts = np.array([[meio, meio-h], [meio+w, meio], [meio, meio+h], [meio-w, meio]])
    for y in range(t):
        for x in range(t):
            if _ponto_em_poligono(x, y, pts):
                img[y, x] = 1.0
    return img

GERADORES = {"estrela": desenhar_estrela, "cruz": desenhar_cruz, "losango": desenhar_losango}

In [ ]:
def gerar_imagem(classe, rng):
    img = GERADORES[classe](TAMANHO, rng)
    ruido = rng.normal(0, 0.05, img.shape).astype(np.float32)
    return np.clip(img + ruido, 0, 1)

def imagem_para_vetor(img):
    return img.flatten()

def gerar_dataset(rng):
    X, y, exemplos = [], [], []
    for classe in CLASSES:
        for i in range(AMOSTRAS):
            img = gerar_imagem(classe, rng)
            X.append(imagem_para_vetor(img))
            y.append(ROTULOS[classe])
            if i == 0:
                exemplos.append((img, classe))
    return np.array(X), np.array(y), exemplos

rng = np.random.default_rng(42)
X, y, exemplos = gerar_dataset(rng)
print(f"Amostras: {X.shape[0]} | Features: {X.shape[1]} | Classes: {CLASSES}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, (img, nome) in zip(axes, exemplos):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(nome.capitalize())
    ax.axis("off")
plt.suptitle("Exemplos de símbolos (entrada visual)")
plt.show()

vetor = imagem_para_vetor(exemplos[0][0])
print(f"Imagem {exemplos[0][0].shape} -> vetor {vetor.shape}")
print(f"Primeiros 10 pixels: {vetor[:10]}")

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

modelo = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10, gamma="scale"))
])
modelo.fit(X_tr, y_tr)
y_pred = modelo.predict(X_te)

print(f"Acurácia: {accuracy_score(y_te, y_pred):.2%}")
print(classification_report(y_te, y_pred, target_names=CLASSES))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_te, y_pred, display_labels=CLASSES, ax=ax, cmap="Blues")
ax.set_title("Matriz de confusão — SVM")
plt.show()

In [ ]:
def classificar(modelo, img):
    return CLASSES[modelo.predict(imagem_para_vetor(img).reshape(1, -1))[0]]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, classe in zip(axes, CLASSES):
    img_nova = gerar_imagem(classe, rng)
    pred = classificar(modelo, img_nova)
    ax.imshow(img_nova, cmap="gray", vmin=0, vmax=1)
    cor = "green" if pred == classe else "red"
    ax.set_title(f"Real: {classe}\nSVM: {pred}", color=cor)
    ax.axis("off")
plt.suptitle("Símbolos novos após o treino")
plt.show()

## Conclusão

O SVM classifica os símbolos usando apenas o **vetor de pixels**. A imagem é convertida em números antes do treino — o modelo nunca "vê" o desenho como humano, apenas os atributos numéricos.